# Job Market Intelligence – Data Cleaning

This notebook cleans and prepares the No Fluff Jobs dataset for exploratory data analysis and machine learning.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nofluff_it_jobs.csv"
)

df = pd.read_csv(RAW_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Dataset shape:", df.shape)

PROJECT_ROOT: G:\pandas\job_market_intelligence
Dataset shape: (3309, 24)


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3309 entries, 0 to 3308
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   job_id                3309 non-null   object 
 1   url                   3309 non-null   object 
 2   title                 3309 non-null   object 
 3   category              3065 non-null   object 
 4   experience            3258 non-null   object 
 5   experience_years_min  1945 non-null   float64
 6   workplace             3009 non-null   object 
 7   job_locations         2038 non-null   object 
 8   company               3309 non-null   object 
 9   company_size          2452 non-null   object 
 10  company_founded       2452 non-null   float64
 11  company_locations     2452 non-null   object 
 12  salary_min            2342 non-null   float64
 13  salary_max            2342 non-null   float64
 14  salary_currency       2342 non-null   object 
 15  contract_type        

## 1. Missing values

Check the number and percentage of missing values in each column.

In [3]:
missing_values = (
    df.isna()
    .sum()
    .to_frame("missing_count")
)

missing_values["missing_percent"] = (
    missing_values["missing_count"]
    / len(df)
    * 100
).round(2)

missing_values = missing_values.sort_values(
    "missing_percent",
    ascending=False
)

missing_values

,missing_count,missing_percent
experience_years_min,1364,41.22
job_locations,1271,38.41
nice_to_have,1199,36.23
salary_max,967,29.22
salary_currency,967,29.22
salary_min,967,29.22
company_locations,857,25.90
company_founded,857,25.90
company_size,857,25.90
contract_type,763,23.06


## 2. Duplicates

Check for duplicated rows, job IDs, and job URLs.

In [4]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate job IDs:", df["job_id"].duplicated().sum())
print("Duplicate URLs:", df["url"].duplicated().sum())

Duplicate rows: 0
Duplicate job IDs: 0
Duplicate URLs: 0


## 3. Categorical values

Inspect the most important categorical columns and identify inconsistent or unexpected values.

In [5]:
columns_to_check = [
    "experience",
    "workplace",
    "category",
    "contract_type",
    "salary_currency",
]

for col in columns_to_check:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(30))


--- experience ---
experience
Senior    1689
Mid       1289
Junior     181
Expert      99
NaN         51
Name: count, dtype: int64

--- workplace ---
workplace
Hybrid    1815
Remote    1184
NaN        300
Onsite      10
Name: count, dtype: int64

--- category ---
category
Backend               290
DevOps                247
NaN                   244
Data                  238
Testing               202
Java, Backend         196
Security              184
ERP                   182
Project Manager       177
Architecture          146
AI                    142
Support                97
Fullstack              94
Python, Data           84
Product Management     78
Frontend               70
.NET, Backend          69
Business Analysis      66
Mobile                 61
Python, Backend        61
Java, Fullstack        60
Python, AI             57
.NET, Fullstack        44
Embedded               40
Python, Testing        35
Python, DevOps         33
Java, Testing          26
Python, Fullstack      2

## 4. Numerical values

Inspect numerical columns and check for invalid or unexpected values.

In [6]:
numerical_columns = [
    "experience_years_min",
    "company_founded",
    "salary_min",
    "salary_max",
]

df[numerical_columns].describe()

,experience_years_min,company_founded,salary_min,salary_max
count,1945.000000,2452.000000,2342.000000,2342.000000
mean,4.754242,2000.829527,21789.706234,26864.597353
std,1.974181,30.541480,7188.903414,8433.717246
min,0.000000,1783.000000,55.000000,70.000000
25%,3.000000,1996.000000,17000.000000,21840.000000
50%,5.000000,2007.000000,21840.000000,26880.000000
75%,5.000000,2015.000000,26040.000000,31920.000000
max,15.000000,2026.000000,50400.000000,67200.000000


In [7]:
print(
    "Salary min > salary max:",
    (df["salary_min"] > df["salary_max"]).sum()
)

print(
    "Salary <= 0:",
    (
        (df["salary_min"] <= 0)
        | (df["salary_max"] <= 0)
    ).sum()
)

print(
    "Company founded > 2026:",
    (df["company_founded"] > 2026).sum()
)

print(
    "Experience years < 0:",
    (df["experience_years_min"] < 0).sum()
)

Salary min > salary max: 0
Salary <= 0: 0
Company founded > 2026: 0
Experience years < 0: 0


In [8]:
low_salary_jobs = df[
    (df["salary_min"] < 5000)
    | (df["salary_max"] < 5000)
][
    [
        "title",
        "company",
        "salary_min",
        "salary_max",
        "salary_currency",
        "contract_type",
        "url",
    ]
]

print("Low salary jobs:", len(low_salary_jobs))

low_salary_jobs

Low salary jobs: 22


,title,company,salary_min,salary_max,salary_currency,contract_type,url
71,AI Coach - Tester,Antal,100.0,140.0,PLN,B2B,https://nofluffjobs.com/pl/job/ai-coach-tester...
86,AI Developer,Verita HR,3360.0,5250.0,PLN,B2B,https://nofluffjobs.com/pl/job/ai-developer-ve...
110,AI Governance Engineer (Microsoft 365 / Copilot),Verita HR,3990.0,4200.0,PLN,B2B,https://nofluffjobs.com/pl/job/ai-governance-e...
133,AI Solution Engineer (Cybersecurity),Antal,180.0,230.0,PLN,B2B,https://nofluffjobs.com/pl/job/ai-solution-eng...
274,Architekt systemowy,Antal,175.0,195.0,PLN,B2B,https://nofluffjobs.com/pl/job/architekt-syste...
314,AWS Python Develper,Antal,180.0,230.0,PLN,B2B,https://nofluffjobs.com/pl/job/aws-python-deve...
363,Business Analyst,ITFS Sp. z o.o.,140.0,155.0,PLN,B2B,https://nofluffjobs.com/pl/job/business-analys...
576,Data Engineer (Hadoop),Verita HR,1450.0,1550.0,PLN,B2B,https://nofluffjobs.com/pl/job/data-engineer-h...
667,Devops/ Cloud Engineer,Verita HR,1450.0,1550.0,PLN,B2B,https://nofluffjobs.com/pl/job/devops-cloud-en...
954,Geospatial Data Software Engineer (Regular/Sen...,Spyrosoft,2040.0,4200.0,EUR,"B2B, Umowa o pracę",https://nofluffjobs.com/pl/job/geospatial-data...


In [9]:
df[["valid_until", "start_date", "scraped_at"]].head(20)

,valid_until,start_date,scraped_at
0,30.08.2026,ASAP,2026-08-13 20:11:15
1,05.09.2026,ASAP,2026-08-13 20:11:17
2,06.09.2026,ASAP,2026-08-13 20:11:19
3,11.09.2026,NaN,2026-08-13 20:11:21
4,10.09.2026,ASAP,2026-08-13 20:11:23
5,10.09.2026,2026-08-11,2026-08-13 20:11:26
6,11.09.2026,ASAP,2026-08-13 20:11:28
7,06.09.2026,ASAP,2026-08-13 20:11:31
8,11.09.2026,ASAP,2026-08-13 20:11:33
9,12.09.2026,ASAP,2026-08-13 20:11:35


In [10]:
start_date_check = (
    df["start_date"]
    .dropna()
    .astype(str)
    .str.len()
)

start_date_check.describe()

count    3116.000000
mean        4.521823
std         1.691022
min         4.000000
25%         4.000000
50%         4.000000
75%         4.000000
max        10.000000
Name: start_date, dtype: float64

In [11]:
df.loc[
    df["start_date"]
    .fillna("")
    .astype(str)
    .str.len()
    .sort_values(ascending=False)
    .head(20)
    .index,
    ["title", "start_date"]
]

,title,start_date
380,Business Operations Specialist (AI & FinTech) ...,2026-09-01
3217,Tester Manualny,2026-07-08
1486,.NET Developer,2026-07-07
3308,Web Developer,2026-09-01
671,DevOps Engineer,2026-05-22
1443,MLOps/LLM Platform Tech Lead,2026-06-09
1230,"Junior Quality Assurance Analyst with Spanish,...",2026-08-24
195,Architekt Danych i AI,2026-05-14
323,Azure DevOps Engineer,2026-05-01
5,Data Analyst,2026-08-11


In [12]:
invalid_start_dates = df[
    df["start_date"].notna()
    & (df["start_date"] != "ASAP")
    & ~df["start_date"].str.match(
        r"^\d{4}-\d{2}-\d{2}$",
        na=False
    )
]

print("Unexpected start_date values:", len(invalid_start_dates))

invalid_start_dates[
    ["title", "start_date"]
].head(20)

Unexpected start_date values: 0


,title,start_date


## 5. Date columns

Convert date-related columns to datetime format and preserve the ASAP information separately.

In [13]:
df["valid_until"] = pd.to_datetime(
    df["valid_until"],
    format="%d.%m.%Y",
    errors="coerce"
)

df["scraped_at"] = pd.to_datetime(
    df["scraped_at"],
    errors="coerce"
)

df["start_asap"] = (
    df["start_date"]
    .eq("ASAP")
)

df["start_date_parsed"] = pd.to_datetime(
    df["start_date"].where(
        df["start_date"] != "ASAP"
    ),
    format="%Y-%m-%d",
    errors="coerce"
)

df[
    [
        "start_date",
        "start_asap",
        "start_date_parsed",
        "valid_until",
        "scraped_at",
    ]
].head(20)

,start_date,start_asap,start_date_parsed,valid_until,scraped_at
0,ASAP,True,NaT,2026-08-30,2026-08-13 20:11:15
1,ASAP,True,NaT,2026-09-05,2026-08-13 20:11:17
2,ASAP,True,NaT,2026-09-06,2026-08-13 20:11:19
3,NaN,False,NaT,2026-09-11,2026-08-13 20:11:21
4,ASAP,True,NaT,2026-09-10,2026-08-13 20:11:23
5,2026-08-11,False,2026-08-11,2026-09-10,2026-08-13 20:11:26
6,ASAP,True,NaT,2026-09-11,2026-08-13 20:11:28
7,ASAP,True,NaT,2026-09-06,2026-08-13 20:11:31
8,ASAP,True,NaT,2026-09-11,2026-08-13 20:11:33
9,ASAP,True,NaT,2026-09-12,2026-08-13 20:11:35


In [14]:
print("valid_until NaT:", df["valid_until"].isna().sum())
print("scraped_at NaT:", df["scraped_at"].isna().sum())

print()
print("ASAP:", df["start_asap"].sum())
print("Parsed start dates:", df["start_date_parsed"].notna().sum())
print("Missing original start_date:", df["start_date"].isna().sum())

print()
print(
    "Total start_date:",
    df["start_asap"].sum()
    + df["start_date_parsed"].notna().sum()
    + df["start_date"].isna().sum()
)

valid_until NaT: 125
scraped_at NaT: 0

ASAP: 2845
Parsed start dates: 271
Missing original start_date: 193

Total start_date: 3309


In [15]:
df["company_size"].value_counts(dropna=False).head(30)

company_size
NaN          857
500 - 999    414
1000+        356
250 - 499    284
100+         278
50 - 249     167
40+          108
+1000         97
500           73
10 - 49       58
1500          51
+6000         48
7500+         36
350+          35
250           32
2000+         31
300+          23
1700+         22
500+          22
5000          20
2400+         19
200+          17
450+          16
+2400         15
50+           15
10+           14
13400+        13
110000        13
600+          13
20+           12
Name: count, dtype: int64

In [16]:
def company_size_format(value):

    if pd.isna(value):
        return "missing"

    value = str(value).strip()

    if re.fullmatch(r"\d+\s*-\s*\d+", value):
        return "range"

    if re.fullmatch(r"\d+\+", value):
        return "min_plus"

    if re.fullmatch(r"\+\d+", value):
        return "plus_min"

    if re.fullmatch(r"\d+", value):
        return "exact"

    return "other"


df["company_size"].apply(
    company_size_format
).value_counts()

company_size
min_plus    1100
range        937
missing      857
exact        232
plus_min     171
other         12
Name: count, dtype: int64

In [17]:
other_company_sizes = df[
    df["company_size"].apply(company_size_format) == "other"
]["company_size"].value_counts()

other_company_sizes

company_size
> 150      9
74,000     2
10 000+    1
Name: count, dtype: int64

## 6. Company size

Standardize company size values and extract minimum and maximum employee counts.

In [18]:
def parse_company_size(value):

    if pd.isna(value):
        return np.nan, np.nan

    value = str(value).strip()

    value = value.replace(",", "")
    value = value.replace(" ", "")

    match = re.fullmatch(r"(\d+)-(\d+)", value)

    if match:
        return (
            int(match.group(1)),
            int(match.group(2)),
        )

    match = re.fullmatch(r"(\d+)\+", value)

    if match:
        return int(match.group(1)), np.nan

    match = re.fullmatch(r"\+(\d+)", value)

    if match:
        return int(match.group(1)), np.nan

    match = re.fullmatch(r">(\d+)", value)

    if match:
        return int(match.group(1)) + 1, np.nan

    if re.fullmatch(r"\d+", value):
        number = int(value)
        return number, number

    return np.nan, np.nan

In [19]:
df[
    ["company_size_min", "company_size_max"]
] = df["company_size"].apply(
    lambda x: pd.Series(
        parse_company_size(x)
    )
)

df[
    [
        "company_size",
        "company_size_min",
        "company_size_max",
    ]
].head(20)

,company_size,company_size_min,company_size_max
0,10 - 49,10.0,49.0
1,NaN,NaN,NaN
2,50 - 249,50.0,249.0
3,NaN,NaN,NaN
4,250 - 499,250.0,499.0
5,500 - 999,500.0,999.0
6,500 - 999,500.0,999.0
7,40+,40.0,NaN
8,500 - 999,500.0,999.0
9,NaN,NaN,NaN


In [20]:
print(
    "Original company size available:",
    df["company_size"].notna().sum()
)

print(
    "Parsed company size:",
    df["company_size_min"].notna().sum()
)

Original company size available: 2452
Parsed company size: 2452


## 7. Numerical data types

Convert integer-like numerical columns to nullable integer format.

In [21]:
df["experience_years_min"] = (
    df["experience_years_min"]
    .astype("Int64")
)

df["company_founded"] = (
    df["company_founded"]
    .astype("Int64")
)

df["company_size_min"] = (
    df["company_size_min"]
    .astype("Int64")
)

df["company_size_max"] = (
    df["company_size_max"]
    .astype("Int64")
)

df[
    [
        "experience_years_min",
        "company_founded",
        "company_size_min",
        "company_size_max",
    ]
].dtypes

experience_years_min    Int64
company_founded         Int64
company_size_min        Int64
company_size_max        Int64
dtype: object

## 8. Text standardization

Remove leading and trailing whitespace and convert empty strings to missing values.

In [22]:
text_columns = df.select_dtypes(
    include="object"
).columns

for col in text_columns:
    df[col] = (
        df[col]
        .str.strip()
        .replace("", pd.NA)
    )

print("Text columns cleaned:", len(text_columns))

Text columns cleaned: 18


In [23]:
empty_strings = (
    df[text_columns]
    .eq("")
    .sum()
    .sum()
)

print("Empty strings remaining:", empty_strings)

Empty strings remaining: 0


## 9. Salary analysis flag

Preserve the original salary values and create a conservative flag for salary records suitable for monthly PLN analysis.

In [24]:
df["salary_analysis_eligible"] = (
    df["salary_min"].notna()
    & df["salary_max"].notna()
    & (df["salary_currency"] == "PLN")
    & (df["salary_min"] >= 5000)
    & (df["salary_max"] >= 5000)
)

print(
    df["salary_analysis_eligible"]
    .value_counts()
)

print(
    "\nEligible salary offers:",
    df["salary_analysis_eligible"].sum()
)

salary_analysis_eligible
True     2320
False     989
Name: count, dtype: int64

Eligible salary offers: 2320


## 10. Final numerical types

Convert salary columns to nullable integer format while preserving missing values.

In [25]:
df["salary_min"] = df["salary_min"].astype("Int64")
df["salary_max"] = df["salary_max"].astype("Int64")

df[
    [
        "salary_min",
        "salary_max",
        "experience_years_min",
        "company_founded",
        "company_size_min",
        "company_size_max",
    ]
].dtypes

salary_min              Int64
salary_max              Int64
experience_years_min    Int64
company_founded         Int64
company_size_min        Int64
company_size_max        Int64
dtype: object

In [26]:
PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEAN_PATH = (
    PROCESSED_DIR
    / "nofluff_it_jobs_clean.csv"
)

In [27]:
import re
import pandas as pd

def clean_experience(row):

    title = str(row["title"]).lower()

    levels_found = []

    if re.search(r"\b(junior|jr)\b", title):
        levels_found.append("Junior")

    if re.search(r"\b(mid|regular)\b", title):
        levels_found.append("Mid")

    if re.search(r"\b(senior|sr)\b", title):
        levels_found.append("Senior")

    if re.search(r"\b(expert|principal|head)\b", title):
        levels_found.append("Expert")

    if len(set(levels_found)) == 1:
        return levels_found[0]

    return row["experience"]


df["experience_clean"] = df.apply(
    clean_experience,
    axis=1
)

In [28]:
changed_mask = (
    df["experience"].fillna("Missing")
    !=
    df["experience_clean"].fillna("Missing")
)

print(
    "Changed experience levels:",
    changed_mask.sum()
)

Changed experience levels: 136


In [29]:
df["experience_clean"].value_counts(
    dropna=False
)

experience_clean
Senior    1751
Mid       1229
Junior     140
Expert     139
NaN         50
Name: count, dtype: int64

In [30]:
def extract_experience_years_v3(text):

    if pd.isna(text):
        return pd.NA

    text = str(text).lower()

    patterns = [
  
        (
            r"(?:minimum|min\.?|co najmniej|od)?\s*"
            r"(\d{1,2})"
            r"\s*"
            r"(?:(?:[-–]|do)\s*\d{1,2}\+?)?"
            r"\s*\+?"
            r"\s*"
            r"(?:lat|lata|rok|roku)"
            r"[^.!?\n]{0,50}"
            r"doświadczen"
        ),


        (
            r"(?:minimum|min\.?|at least)?\s*"
            r"(\d{1,2})"
            r"\s*"
            r"(?:(?:[-–]|to)\s*\d{1,2}\+?)?"
            r"\s*\+?"
            r"\s*"
            r"(?:years?|yrs?)"
            r"[^.!?\n]{0,50}"
            r"experience"
        ),

        (
            r"experience"
            r"[^.!?\n]{0,30}"
            r"(\d{1,2})"
            r"\s*\+?"
            r"\s*(?:years?|yrs?)"
        ),
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            text,
            re.IGNORECASE
        )

        if match:
            return int(match.group(1))

    return pd.NA

In [31]:
df["experience_years_v3"] = (
    df["requirements"]
    .apply(extract_experience_years_v3)
    .astype("Int64")
)

In [32]:
df["experience_years_confident"] = (
    df["experience_years_min"]
    .where(
        df["experience_years_v3"].notna()
        & (
            df["experience_years_min"]
            == df["experience_years_v3"]
        )
    )
    .astype("Int64")
)

In [33]:
df = df.drop(
    columns=["experience_years_v3"]
)

In [34]:
df = df.drop(columns=["experience"])

df = df.rename(
    columns={
        "experience_clean": "experience"
    }
)

In [35]:
df["experience"].value_counts(dropna=False)

experience
Senior    1751
Mid       1229
Junior     140
Expert     139
NaN         50
Name: count, dtype: int64

In [36]:
df = df.drop(
    columns=[
        "experience_years_min",
    ],
    errors="ignore"
)

In [37]:
df = df.rename(
    columns={
        "experience_years_confident": "experience_years_min"
    }
)

In [38]:
print("Dataset shape:", df.shape)

print()
print(df["experience"].value_counts(dropna=False))

print()
print(
    "Experience years available:",
    df["experience_years_min"].notna().sum()
)

print(
    "Duplicate URLs:",
    df["url"].duplicated().sum()
)

print(
    "Missing titles:",
    df["title"].isna().sum()
)

Dataset shape: (3309, 29)

experience
Senior    1751
Mid       1229
Junior     140
Expert     139
NaN         50
Name: count, dtype: int64

Experience years available: 1729
Duplicate URLs: 0
Missing titles: 0


In [39]:
df.to_csv(
    CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", CLEAN_PATH)

Saved: G:\pandas\job_market_intelligence\data\processed\nofluff_it_jobs_clean.csv
